# Phase 1 - AwA2 Manifest Preparation

This notebook mirrors `scripts/prepare_awa2.py` in small, explicit steps. It prepares the AwA2 image manifest and the stable `class_name -> label` mapping used by the PyTorch Dataset.

## What this phase does

- finds `JPEGImages/` under `data/AWA2/`;
- optionally downloads the AwA2 archive if the images are missing;
- scans class folders;
- builds reproducible train/validation/test splits;
- writes a manifest with `filepath,label,class_name,split`;
- writes a stable class mapping CSV.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from scripts.prepare_awa2 import (
    DEFAULT_AWA2_URL,
    apply_subset,
    collect_class_images,
    download_archive,
    extract_archive,
    find_jpeg_images_dir,
    write_manifests,
)
from src.utils import set_seed

PROJECT_ROOT

PosixPath('/home/emma/DeepLearning/Deep_Learning_XAI')

## Configuration

Use the debug subset while developing. Switch `USE_DEBUG_SUBSET` to `False` for the full 50-class manifest.

In [2]:
DATA_ROOT = PROJECT_ROOT / "data" / "AWA2"
MANIFEST_DIR = DATA_ROOT

SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

DOWNLOAD_IF_MISSING = False
USE_DEBUG_SUBSET = True
MAX_CLASSES = 10
MAX_IMAGES_PER_CLASS = 200

manifest_name = "awa2_manifest_debug.csv" if USE_DEBUG_SUBSET else "awa2_manifest.csv"
class_map_name = "class_to_idx_debug.csv" if USE_DEBUG_SUBSET else "class_to_idx.csv"

set_seed(SEED)
DATA_ROOT

PosixPath('/home/emma/DeepLearning/Deep_Learning_XAI/data/AWA2')

## Locate or download the images

By default this notebook does not download AwA2. Set `DOWNLOAD_IF_MISSING = True` only when you intentionally want to fetch the full archive.

In [3]:
try:
    jpeg_dir = find_jpeg_images_dir(DATA_ROOT)
except FileNotFoundError:
    if not DOWNLOAD_IF_MISSING:
        raise
    archive_path = DATA_ROOT / "AwA2-data.zip"
    download_archive(DEFAULT_AWA2_URL, archive_path)
    extract_archive(archive_path, DATA_ROOT)
    jpeg_dir = find_jpeg_images_dir(DATA_ROOT)

jpeg_dir

PosixPath('/home/emma/DeepLearning/Deep_Learning_XAI/data/AWA2/Animals_with_Attributes2/JPEGImages')

## Collect images and optionally build a debug subset

Classes are sorted by folder name before labels are assigned. This keeps the mapping deterministic.

In [4]:
class_to_images = collect_class_images(jpeg_dir)

if USE_DEBUG_SUBSET:
    class_to_images = apply_subset(
        class_to_images=class_to_images,
        max_classes=MAX_CLASSES,
        max_images_per_class=MAX_IMAGES_PER_CLASS,
        seed=SEED,
    )

num_images = sum(len(paths) for paths in class_to_images.values())
num_classes = len(class_to_images)
num_classes, num_images

(10, 1967)

## Write the manifest and class mapping

The split is done inside each class, so each split keeps the selected class distribution balanced.

In [5]:
manifest_path, class_map_path = write_manifests(
    class_to_images=class_to_images,
    manifest_dir=MANIFEST_DIR,
    manifest_name=manifest_name,
    class_map_name=class_map_name,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=SEED,
)

manifest_path, class_map_path

(PosixPath('/home/emma/DeepLearning/Deep_Learning_XAI/data/AWA2/awa2_manifest_debug.csv'),
 PosixPath('/home/emma/DeepLearning/Deep_Learning_XAI/data/AWA2/class_to_idx_debug.csv'))

## Quick inspection

In [6]:
import pandas as pd

manifest = pd.read_csv(manifest_path)
class_map = pd.read_csv(class_map_path)

display(manifest.head())
display(manifest["split"].value_counts())
display(class_map.head(10))

,filepath,label,class_name,split
0,/home/emma/DeepLearning/Deep_Learning_XAI/data...,0,antelope,train
1,/home/emma/DeepLearning/Deep_Learning_XAI/data...,0,antelope,train
2,/home/emma/DeepLearning/Deep_Learning_XAI/data...,0,antelope,train
3,/home/emma/DeepLearning/Deep_Learning_XAI/data...,0,antelope,train
4,/home/emma/DeepLearning/Deep_Learning_XAI/data...,0,antelope,train


split
train    1376
test      297
val       294
Name: count, dtype: int64

,class_name,label
0,antelope,0
1,bat,1
2,beaver,2
3,blue+whale,3
4,bobcat,4
5,buffalo,5
6,chihuahua,6
7,chimpanzee,7
8,collie,8
9,cow,9
